In [1]:
import pandas as pd
import numpy as np
import os, krippendorff
from collections import Counter

from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import cohen_kappa_score, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.metrics.pairwise import cosine_similarity

from dotenv import load_dotenv

load_dotenv()

True

In [2]:
df_extraction_annotation = pd.read_csv("annotation_work/extraction_annotation.csv", sep=",")
df_classification_annotation = pd.read_csv("annotation_work/classification_annotation.csv", sep=",")

gold_statements = pd.read_csv("input_data/gold_statements.csv", sep=",")
linguistic_statements = pd.read_csv("paper_results/linguistic_statements.csv", sep=",")

# 1. Inter-annotator agreement

In [3]:
def compute_agreement(df, a1, a2):
    sub = df[[a1, a2]].dropna()

    y1 = sub[a1]
    y2 = sub[a2]

    percent_agreement = (y1 == y2).mean()
    kappa = cohen_kappa_score(y1, y2)

    labels = sorted(set(y1.unique()) | set(y2.unique()))
    cm = confusion_matrix(y1, y2, labels=labels)

    return percent_agreement, kappa, labels, cm, len(sub)

def encode_column(col, encoder):
    encoded = pd.Series(np.nan, index=col.index)
    mask = col.notna()
    encoded[mask] = encoder.transform(col[mask])
    return encoded

def compute_krippendorff_alpha(df, annotators):
    sub = df[annotators].dropna()
    encoder = LabelEncoder()
    all_labels = pd.concat([sub[a] for a in annotators]).dropna()
    encoder.fit(all_labels)
    encoded_data = np.array([encode_column(sub[a], encoder) for a in annotators])
    alpha = krippendorff.alpha(encoded_data, level_of_measurement='nominal')   
    return alpha

def compute_three_way_agreement(df, annotators):
    sub = df[annotators].dropna()
    n = len(sub)
    if n == 0:
        return 0.0, 0
    all_agree = sub.apply(lambda row: len(set(row)) == 1, axis=1)
    pct = all_agree.mean()
    print(f"Three-way agreement ({', '.join(annotators)}):")
    print(f"  Items annotated by all three: {n}")
    print(f"  All three agree: {all_agree.sum()} / {n} ({pct:.2%})")
    return pct, n

def compute_all_agreements(df, annotators):
    results = {}
    for i in range(len(annotators)):
        for j in range(i + 1, len(annotators)):
            a1 = annotators[i]
            a2 = annotators[j]
            percent_agreement, kappa, labels, cm, n = compute_agreement(df, a1, a2)
            results[(a1, a2)] = {
                'percent_agreement': percent_agreement,
                'kappa': kappa,
                'labels': labels,
                'confusion_matrix': cm,
                'n': n
            }
            print(f"Agreement between {a1} and {a2}:")
            print(f"  Percent agreement: {percent_agreement:.2%}")
            print(f"  Cohen's kappa: {kappa:.3f}")
            print(f"  Labels: {labels}")
            print(f"  Confusion matrix:\n{cm}\n")
    return results

## 1.1. Extraction task

In [4]:
annotators = ["annotator1", "annotator2", "annotator3"]
alpha = compute_krippendorff_alpha(df_extraction_annotation, annotators)
print("\nKrippendorff's alpha:", alpha)
three = compute_three_way_agreement(df_extraction_annotation, annotators)
pairs = compute_all_agreements(df_extraction_annotation, annotators)


Krippendorff's alpha: 0.29949534161490676
Three-way agreement (annotator1, annotator2, annotator3):
  Items annotated by all three: 134
  All three agree: 89 / 134 (66.42%)
Agreement between annotator1 and annotator2:
  Percent agreement: 81.34%
  Cohen's kappa: 0.397
  Labels: ['ok', 'problem']
  Confusion matrix:
[[96  9]
 [16 13]]

Agreement between annotator1 and annotator3:
  Percent agreement: 73.13%
  Cohen's kappa: 0.208
  Labels: ['ok', 'problem']
  Confusion matrix:
[[87 18]
 [18 11]]

Agreement between annotator2 and annotator3:
  Percent agreement: 78.36%
  Cohen's kappa: 0.301
  Labels: ['ok', 'problem']
  Confusion matrix:
[[94 18]
 [11 11]]



## 1.2. Classification task: human annotators

In [5]:
annotators = ["annotator1_class", "annotator2_class", "annotator3_class"]
alpha = compute_krippendorff_alpha(df_classification_annotation, annotators)
print("\nKrippendorff's alpha:", alpha)
compute_three_way_agreement(df_classification_annotation, annotators)
pairs = compute_all_agreements(df_classification_annotation, annotators)


Krippendorff's alpha: 0.7499031466574835
Three-way agreement (annotator1_class, annotator2_class, annotator3_class):
  Items annotated by all three: 194
  All three agree: 146 / 194 (75.26%)
Agreement between annotator1_class and annotator2_class:
  Percent agreement: 81.44%
  Cohen's kappa: 0.725
  Labels: ['C', 'L-Spec', 'L-Theo', 'L-Typo', 'Other', 'S']
  Confusion matrix:
[[35  1  0  0  0  0]
 [ 1 82  0  0  0  0]
 [ 1 16 32  4  0  4]
 [ 0  1  0  4  0  0]
 [ 0  1  0  0  0  0]
 [ 1  6  0  0  0  5]]

Agreement between annotator1_class and annotator3_class:
  Percent agreement: 87.63%
  Cohen's kappa: 0.825
  Labels: ['C', 'L-Spec', 'L-Theo', 'L-Typo', 'Other', 'S']
  Confusion matrix:
[[35  0  1  0  0  0]
 [ 0 75  2  4  0  2]
 [ 1  3 47  5  0  1]
 [ 0  0  2  3  0  0]
 [ 0  0  0  0  0  1]
 [ 1  1  0  0  0 10]]

Agreement between annotator2_class and annotator3_class:
  Percent agreement: 79.38%
  Cohen's kappa: 0.700
  Labels: ['C', 'L-Spec', 'L-Theo', 'L-Typo', 'S']
  Confusion matri

## 1.3. Classification task: human annotators + models

In [6]:
annotators = ["annotator1_class", "annotator2_class", "annotator3_class", "gemini_class", "openai_class"]
alpha = compute_krippendorff_alpha(df_classification_annotation, annotators)
print("\nKrippendorff's alpha:", alpha)
compute_three_way_agreement(df_classification_annotation, annotators)
pairs = compute_all_agreements(df_classification_annotation, annotators)


Krippendorff's alpha: 0.662829619787408
Three-way agreement (annotator1_class, annotator2_class, annotator3_class, gemini_class, openai_class):
  Items annotated by all three: 194
  All three agree: 112 / 194 (57.73%)
Agreement between annotator1_class and annotator2_class:
  Percent agreement: 81.44%
  Cohen's kappa: 0.725
  Labels: ['C', 'L-Spec', 'L-Theo', 'L-Typo', 'Other', 'S']
  Confusion matrix:
[[35  1  0  0  0  0]
 [ 1 82  0  0  0  0]
 [ 1 16 32  4  0  4]
 [ 0  1  0  4  0  0]
 [ 0  1  0  0  0  0]
 [ 1  6  0  0  0  5]]

Agreement between annotator1_class and annotator3_class:
  Percent agreement: 87.63%
  Cohen's kappa: 0.825
  Labels: ['C', 'L-Spec', 'L-Theo', 'L-Typo', 'Other', 'S']
  Confusion matrix:
[[35  0  1  0  0  0]
 [ 0 75  2  4  0  2]
 [ 1  3 47  5  0  1]
 [ 0  0  2  3  0  0]
 [ 0  0  0  0  0  1]
 [ 1  1  0  0  0 10]]

Agreement between annotator1_class and gemini_class:
  Percent agreement: 73.71%
  Cohen's kappa: 0.630
  Labels: ['C', 'L-Spec', 'L-Theo', 'L-Typo',

In [7]:
human_annotators = ["annotator1_class", "annotator2_class", "annotator3_class"]
all_annotators = human_annotators + ["gemini_class", "openai_class"]

final_class = []
all_agree = 0
majority_agree = 0
no_agreement = 0
no_ann = 0

for idx, row in df_classification_annotation.iterrows():
    votes = [row[ann] for ann in human_annotators]
    vote_counts = Counter(votes)
    most_common = vote_counts.most_common(1)[0]

    if most_common[0] == None:
        no_ann += 1
        model_votes = [row[ann] for ann in all_annotators if ann not in human_annotators]
        vote_model_counts = Counter(model_votes)
        most_common = vote_model_counts.most_common(1)[0]
        df_classification_annotation.at[idx, 'Gold Class'] = most_common[0]

    elif most_common[1] == 3 and most_common[0] is not None:
        all_agree += 1
        df_classification_annotation.at[idx, 'Gold Class'] = most_common[0]

    # Mayority vote among humans if at least 2 agree, else use model votes
    else:

        model_votes = [row[ann] for ann in all_annotators]
        vote_model_counts = Counter(model_votes)
        most_common = vote_model_counts.most_common(1)[0]
        if most_common[1] >= 3:
            majority_agree += 1
            df_classification_annotation.at[idx, 'Gold Class'] = most_common[0]
        else: 
            no_agreement += 1
            df_classification_annotation.at[idx, 'Gold Class'] = None

print(f"Complete agreement: {all_agree}")
print(f"Majority agreement: {majority_agree}")
print(f"No agreement: {no_agreement}")
print(f"No annotation: {no_ann}")

Complete agreement: 294
Majority agreement: 36
No agreement: 12
No annotation: 0


# 2. Statistics

In [8]:
print("Gold dataset statistics:")
print("Number of statements:", len(gold_statements))
print("Gold Class distribution: \n")
print(gold_statements["Gold Class"].value_counts())

Gold dataset statistics:
Number of statements: 342
Gold Class distribution: 

Gold Class
L-Spec    115
L-Theo    110
C          91
S          16
L-Typo     10
Name: count, dtype: int64


In [9]:
print("Linguistic dataset statistics:")
print("Number of statements:", len(linguistic_statements))
print("Gold Class distribution: \n")
print(linguistic_statements["Gold Class"].value_counts())
print("Gold Verification distribution: \n")
print(linguistic_statements["Gold Verification"].value_counts())

Linguistic dataset statistics:
Number of statements: 681
Gold Class distribution: 

Gold Class
L-Spec    333
L-Theo    318
L-Typo     30
Name: count, dtype: int64
Gold Verification distribution: 

Gold Verification
False    446
True     235
Name: count, dtype: int64


# 3. Tasks results

## 3.1. Task 1: Statement Extraction

In [10]:
def compute_metric(model, gold_df, pred_df):

    # Encode the claims to get embeddings
    gold_embeddings = model.encode(gold_df['statement'].tolist(), convert_to_tensor=True)
    pred_embeddings = model.encode(pred_df['predicted_statement'].tolist(), convert_to_tensor=True)
    # Compute the cosine similarity matrix between gold and prediction embeddings
    cosine_matrix = cosine_similarity(gold_embeddings.cpu().numpy(), pred_embeddings.cpu().numpy())
    # Recall calculation (for each gold claim, find the best predicted match)
    recall_scores = cosine_matrix.max(axis=1)
    avg_recall = np.mean(recall_scores)
    # Precision calculation (for each predicted claim, find the best gold match)
    precision_scores = cosine_matrix.max(axis=0)
    avg_precision = np.mean(precision_scores)
    # F1-score calculation
    f1_score = 2 * (avg_precision * avg_recall) / (avg_precision + avg_recall) if (avg_precision + avg_recall) > 0 else 0
    return [avg_precision, avg_recall, f1_score]

In [11]:
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
precision_scores = []
recall_scores = []
f1_scores = []

directory = "paper_results/experiments_stm_ext"
for exp_name in os.listdir(directory):

    model_name = exp_name.split('.')[0].replace("predictions_", "").replace("_", "-").replace("accounts-fireworks-models-", "")
    pred_df = pd.read_csv(directory + "/" + exp_name, header=0)
    precision, recall, f1 = compute_metric(model, gold_statements, pred_df)
    precision_scores.append([model_name, round(precision, 3)])    
    recall_scores.append([model_name, round(recall, 3)])  
    f1_scores.append([model_name, round(f1, 3)])

table = pd.DataFrame(precision_scores, columns=['Model:', 'Precision'])
table['Recall'] = [recall for _, recall in recall_scores]
table['F1'] = [f1 for _, f1 in f1_scores]
print(table)

                   Model:  Precision  Recall     F1
0              gpt-5-nano      0.873   0.743  0.803
1           deepseek-v3p1      0.965   0.494  0.653
2               kimi-k2p5      0.904   0.854  0.878
3              gpt-5-mini      0.916   0.856  0.885
4   kimi-k2-instruct-0905      0.911   0.772  0.836
5  gemini-3-flash-preview      0.907   0.883  0.894
6        gemini-2-5-flash      0.917   0.853  0.884
7                   gpt-5      0.912   0.861  0.886
8                 gpt-5-2      0.906   0.870  0.888


## 3.2. Task 2: Statement Classification

In [15]:
precision_scores = []
recall_scores = []
f1_scores = []

directory = "paper_results/experiments_class"
for exp_name in os.listdir(directory):
    if not exp_name.endswith(".csv"):
        continue

    model_name = exp_name.split('.')[0].replace("predictions_", "").replace("_", "-").replace("accounts-fireworks-models-", "")
    pred_df = pd.read_csv(os.path.join(directory, exp_name), header=0)

    # Merge on statement text to align gold and predicted classes
    merged = gold_statements[['statement', 'Gold Class']].merge(
        pred_df[['statement', 'predicted_class']],
        on='statement', how='inner'
    ).dropna(subset=['Gold Class', 'predicted_class'])

    if len(merged) == 0:
        print(f"{model_name}: no matching statements, skipping")
        continue

    y_true = merged['Gold Class']
    y_pred = merged['predicted_class']

    precision = precision_score(y_true, y_pred, average='macro', zero_division=0)
    recall = recall_score(y_true, y_pred, average='macro', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)

    precision_scores.append([model_name, round(precision, 3)])
    recall_scores.append([model_name, round(recall, 3)])
    f1_scores.append([model_name, round(f1, 3)])

# Build results table
table = pd.DataFrame(precision_scores, columns=['Model', 'Precision'])
table['Recall'] = [r for _, r in recall_scores]
table['F1'] = [f for _, f in f1_scores]
table = table.sort_values('F1', ascending=False).reset_index(drop=True)
print(table)

                     Model  Precision  Recall     F1
0   gemini-3-flash-preview      0.824   0.890  0.840
1                  gpt-5-2      0.789   0.883  0.811
2            deepseek-v3p1      0.774   0.870  0.805
3                    glm-5      0.774   0.821  0.790
4               gpt-5-nano      0.747   0.799  0.767
5         gemini-2-5-flash      0.715   0.846  0.750
6    kimi-k2-instruct-0905      0.724   0.788  0.744
7                kimi-k2p5      0.733   0.770  0.742
8                    gpt-5      0.696   0.785  0.723
9             minimax-m2p5      0.697   0.826  0.722
10              gpt-5-mini      0.679   0.683  0.680
11           deepseek-v3p2      0.167   0.003  0.006


## 3.3. Task 3: Linguistic Knowledge Evaluation

In [16]:
path_experiments = "paper_results/experiments_fact_check/"
y_true = linguistic_statements["Gold Verification"].astype(bool)

results = []
for file in os.listdir(path_experiments):
    if file.endswith('.csv'):
        df = pd.read_csv(os.path.join(path_experiments, file))
        y_pred = df["veredict"].astype(bool)
        accuracy = accuracy_score(y_true, y_pred)
        precision = precision_score(y_true, y_pred, zero_division=0)
        recall = recall_score(y_true, y_pred, zero_division=0)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        results.append({
            "model": file.replace('.csv', ''),
            "Samples": len(df),
            "Accuracy": accuracy,
            "Precision": precision,
            "Recall": recall,
            "F1": f1
        })

results_df = pd.DataFrame(results).sort_values("F1", ascending=False)
results_df

,model,Samples,Accuracy,Precision,Recall,F1
4,gemini-3-flash-preview,681,0.718062,0.566154,0.782979,0.657143
5,kimi-k2p5,681,0.679883,0.525680,0.740426,0.614841
0,kimi-k2-instruct-0905,681,0.643172,0.489637,0.804255,0.608696
1,deepseek-v3p1,681,0.646109,0.491758,0.761702,0.597663
6,gemini-2.5-flash,681,0.649046,0.494286,0.736170,0.591453
2,gpt-5-mini,681,0.703377,0.566802,0.595745,0.580913
3,gpt-5-nano,681,0.696035,0.575269,0.455319,0.508314


In [17]:
all_results = []

for file in os.listdir(path_experiments):
    if not file.endswith(".csv"):
        continue
    
    model_name = file.replace('.csv', '')
    df_eval = pd.read_csv(os.path.join(path_experiments, file))
    if 'verdict' not in df_eval.columns and 'veredict' in df_eval.columns:
        df_eval['verdict'] = df_eval['veredict']
    df_eval["Gold Verification"] = df_eval["Gold Verification"].astype(bool)
    df_eval["verdict"] = df_eval["verdict"].astype(bool)
    for gold_class, group in df_eval.groupby("Gold Class"):
        y_true = group["Gold Verification"]
        y_pred = group["verdict"]
        
        all_results.append({
            "Model": model_name,
            "Gold Class": gold_class,
            "Samples": len(group),
            "Accuracy": accuracy_score(y_true, y_pred),
            "Precision": precision_score(y_true, y_pred, zero_division=0),
            "Recall": recall_score(y_true, y_pred, zero_division=0),
            "F1": f1_score(y_true, y_pred, zero_division=0)
        })

summary_df = pd.DataFrame(all_results)
summary_df

,Model,Gold Class,Samples,Accuracy,Precision,Recall,F1
0,kimi-k2-instruct-0905,L-Spec,333,0.606607,0.459596,0.791304,0.581470
1,kimi-k2-instruct-0905,L-Theo,318,0.685535,0.530120,0.800000,0.637681
2,kimi-k2-instruct-0905,L-Typo,30,0.600000,0.454545,1.000000,0.625000
3,deepseek-v3p1,L-Spec,333,0.618619,0.466667,0.730435,0.569492
4,deepseek-v3p1,L-Theo,318,0.679245,0.524390,0.781818,0.627737
5,deepseek-v3p1,L-Typo,30,0.600000,0.450000,0.900000,0.600000
6,gpt-5-mini,L-Spec,333,0.651652,0.495935,0.530435,0.512605
7,gpt-5-mini,L-Theo,318,0.751572,0.639640,0.645455,0.642534
8,gpt-5-mini,L-Typo,30,0.766667,0.615385,0.800000,0.695652
9,gpt-5-nano,L-Spec,333,0.627628,0.449438,0.347826,0.392157
